# YOLOv13-S Baseline - AFB / Tuberculosis6208 (Chen split)

Mirror struktur `yolo12.ipynb` (wavelet-yolo12) supaya **apples-to-apples** vs YOLOv12s baseline.

**Setup:**
- Dataset zip di Drive: `MyDrive/Tuberculosis6208.zip` (Pascal-VOC).
- Split: **Chen et al. IJAI 2024** - 1024/140/101, `SPLIT_SEED=42` deterministic.
- Training: 1 run, `MODEL=yolov13s`, `SEED=42`, 60 epoch.
- Logging: **W&B** - project `afb_yolov13_chen`.
- Runtime: A100 ~ 25-30 menit per run.

Notebook portable Colab + local Windows (cells Colab-only auto-skip jika tidak terdeteksi).

## 0. Environment detection

In [1]:
import sys
IS_COLAB = 'google.colab' in sys.modules
print('Environment :', 'Colab' if IS_COLAB else 'local')

Environment : Colab


## 1. Mount Drive (Colab only)

In [2]:
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('skip: not Colab')

MessageError: Error: credential propagation was unsuccessful

## 2. Clone repo (afb-yolo13) + YOLOv13 fork

**Colab:** clone fresh ke `/content/`. Cleanup cache supaya tidak konflik dgn previous run.

**Local:** asumsi `D:/Project/afb-yolo13` dan `D:/Project/yolov13` sudah ada.

In [ ]:
import os, gc
from pathlib import Path
import torch

if IS_COLAB:
    REPO_DIR    = Path('/content/afb-yolo13')
    YOLOV13_DIR = Path('/content/yolov13')

    # Cleanup caches (mirror yolo12.ipynb)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    !find /content -type d -name '__pycache__' -exec rm -rf {} + 2>/dev/null
    !find /content -type f -name '*.pyc' -delete 2>/dev/null
    !pip cache purge -q
    !rm -rf ~/.cache/ultralytics ~/.config/Ultralytics /root/.cache 2>/dev/null

    # afb-yolo13 (scripts + notebook)
    if REPO_DIR.exists():
        !cd {REPO_DIR} && git fetch origin && git checkout main && git pull --ff-only
    else:
        !git clone https://github.com/iswantosan/afb-yolo13.git {REPO_DIR}

    # YOLOv13 fork (iMoonLab)
    if YOLOV13_DIR.exists():
        !cd {YOLOV13_DIR} && git pull --ff-only
    else:
        !git clone https://github.com/iMoonLab/yolov13.git {YOLOV13_DIR}

    os.chdir(YOLOV13_DIR)
    sys.path.insert(0, str(YOLOV13_DIR))
    sys.path.insert(0, str(REPO_DIR))
    print('cwd:', os.getcwd())
    !cd {YOLOV13_DIR} && git log -1 --oneline
else:
    REPO_DIR    = Path('D:/Project/afb-yolo13')
    YOLOV13_DIR = Path('D:/Project/yolov13')
    print('Local repos:')
    print('  REPO_DIR   :', REPO_DIR, '(exists)' if REPO_DIR.exists() else '(MISSING)')
    print('  YOLOV13_DIR:', YOLOV13_DIR, '(exists)' if YOLOV13_DIR.exists() else '(MISSING)')

## 3. Install dependencies + apply L3 patches

Patch script copies custom AFB modules (RodDSC3k2, SpatialFullPAD_Tunnel, HyperACEScale, etc.) ke YOLOv13 source tree. Idempotent — skip kalau sudah ter-patch. Buat backup `.orig` di first apply.

In [ ]:
if IS_COLAB:
    # 1. Apply AFB-YOLOv13 patches FIRST (before pip install -e)
    !python {REPO_DIR}/scripts/apply_yolov13_patches.py {YOLOV13_DIR}
    # 2. Install YOLOv13 editable + wandb
    !pip -q install -e {YOLOV13_DIR} wandb
    !pip -q install -r {REPO_DIR}/requirements.txt
    # 3. (Optional) Install mamba-ssm for Mamba variant. Skip-if-fail safe.
    print('\n[Optional] Installing mamba-ssm for Mamba variant...')
    !pip -q install causal-conv1d>=1.4.0 mamba-ssm>=2.2.0 2>&1 | tail -5 || echo "[warn] mamba-ssm install failed; Mamba variant will use fallback Conv1d"
else:
    print('Local: pastikan sudah jalankan:')
    print(f'  python {REPO_DIR}/scripts/apply_yolov13_patches.py {YOLOV13_DIR}')
    print(f'  pip install -e {YOLOV13_DIR}')
    print(f'  pip install -r {REPO_DIR}/requirements.txt')
    print(f'  (optional) pip install causal-conv1d mamba-ssm  # for Mamba variant')

## 4. Build dataset split

Pilih mode via variable `SPLIT_MODE`:
- `"single"` — 80/10/10 train/val/test (proper standard split, deterministic seed=42)
- `"5fold"` — 5-fold cross-validation (each image in val of exactly 1 fold, no separate test)

5-fold mode = standard untuk medical imaging paper. Reviewer Q2 di MDPI/Sensors/Symmetry biasanya expect ini.

In [ ]:
# === Pilih mode split ===
SPLIT_MODE = "5fold"        # "single" atau "5fold"
SPLIT_SEED = 42

if IS_COLAB:
    DRIVE_ZIP   = '/content/drive/MyDrive/Tuberculosis6208.zip'
    EXTRACT_DIR = '/content/dataset/raw'
    DATASET_SRC = f'{EXTRACT_DIR}/tuberculosis-phonecamera'
else:
    DRIVE_ZIP   = None
    DATASET_SRC = 'D:/project/yolov12/Tuberculosis6208/tuberculosis-phonecamera'

SPLIT_DIR = (f'/content/tb_{SPLIT_MODE}_seed{SPLIT_SEED}' if IS_COLAB
             else f'D:/datasets/tb_{SPLIT_MODE}_seed{SPLIT_SEED}')

# Build kalau belum ada
marker = Path(SPLIT_DIR) / ('data.yaml' if SPLIT_MODE == 'single' else 'fold_0/data.yaml')
if not marker.exists():
    cmd_parts = [
        'python', f'{REPO_DIR}/scripts/build_split.py',
        '--src', f'"{DATASET_SRC}"',
        '--out', f'"{SPLIT_DIR}"',
        '--mode', SPLIT_MODE,
        '--seed', str(SPLIT_SEED),
    ]
    if DRIVE_ZIP and Path(DRIVE_ZIP).exists():
        cmd_parts += ['--zip', f'"{DRIVE_ZIP}"', '--extract-dir', f'"{EXTRACT_DIR}"']
    cmd = ' '.join(cmd_parts)
    print(cmd, '\n')
    os.system(cmd)
else:
    print(f'Split already exists at {SPLIT_DIR}')

# Discover data.yaml paths (list — 1 entry for single, 5 for 5fold)
if SPLIT_MODE == 'single':
    DATA_YAMLS = [str(Path(SPLIT_DIR) / 'data.yaml')]
else:
    DATA_YAMLS = sorted(str(p) for p in Path(SPLIT_DIR).glob('fold_*/data.yaml'))

print(f'\nMode: {SPLIT_MODE}  Folds/Splits: {len(DATA_YAMLS)}')
print('First yaml:')
print(Path(DATA_YAMLS[0]).read_text())

## 5. W&B login

In [ ]:
import wandb
wandb.login()

## 6. Config run

Ganti `MODEL_CFG` untuk varian:

**Baseline (Ultralytics stock):**
- `yolov13n.yaml` / `yolov13s.yaml` / `yolov13l.yaml` / `yolov13x.yaml`

**L1 hyperparam tweak (bukan novelty):**

| YAML | Strategy | Status |
|---|---|---|
| `yolov13s-fine.yaml` | head DSC3k2 k2=5 | 🟡 |
| `yolov13s-he16.yaml` | HyperACE num_hyperedges 8→16 | 🟡 |

**L2 connectivity (risky):**

| YAML | Status |
|---|---|
| `yolov13s-p2.yaml` | ✗ broke gates #6/#7 |

**L3 new module (engineering):**

| YAML | Module |
|---|---|
| `yolov13s-rod.yaml` | RodDSC3k2 |
| `yolov13s-spgate.yaml` | SpatialFullPAD_Tunnel |
| `yolov13s-scfuse.yaml` | HyperACEScale |

**L4 — HyperMIL (real novelty, image-level supervision):**

| Aktivasi | Cara |
|---|---|
| `USE_HYPERMIL = True` di cell ini + base yaml apa pun | Tambah aux MIL head + count loss tanpa ubah YAML |

HyperMIL: hypergraph-backed MIL aux head reading from HyperACE output. Target: label noise (66% far FP yang ternyata mostly real bacilli). Image-level count loss memaksa model discover all bacilli, bukan hanya GT-marked subset.

Run name auto = `<stem>_seed<S>_<EP>ep`, plus `_mil` suffix kalau HyperMIL aktif.

In [ ]:
# Pilih satu (uncomment yang mau dijalankan):
# === baseline ===
MODEL_CFG = 'yolov13s.yaml'                                         # baseline
# === L1 / L3 variants ===
MODEL_CFG = str(REPO_DIR / 'configs' / 'yolov13s-noP5.yaml')
# MODEL_CFG = str(REPO_DIR / 'configs' / 'yolov13s-he16.yaml')
# MODEL_CFG = str(REPO_DIR / 'configs' / 'yolov13s-rod.yaml')
# MODEL_CFG = str(REPO_DIR / 'configs' / 'yolov13s-spgate.yaml')
# MODEL_CFG = str(REPO_DIR / 'configs' / 'yolov13s-scfuse.yaml')

# === L4 HyperMIL toggle ===
USE_HYPERMIL    = False       # set True to enable HyperMIL aux head + count loss
MIL_WEIGHT      = 0.5         # weight for MIL count loss term
MIL_HIDDEN      = 128         # hidden dim for attention pooling + MLP
CONSIST_WEIGHT  = 0.0         # detection-MIL consistency regularizer (0 = off)

PRETRAINED    = 'yolov13s.pt'        # selalu yolov13s.pt (auto-download iMoonLab)
SEED          = 42
EPOCHS        = 60
IMGSZ         = 640
BATCH         = 16
DEVICE        = 0

WANDB_PROJECT = 'afb_yolov13_chen'
RUN_PROJECT   = '/content/runs/afb_yolov13' if IS_COLAB else 'D:/runs/afb_yolov13'
RUN_NAME      = f"{Path(MODEL_CFG).stem}_seed{SEED}_{EPOCHS}ep"
if USE_HYPERMIL:
    RUN_NAME += f'_mil{MIL_WEIGHT:g}'
    if CONSIST_WEIGHT > 0:
        RUN_NAME += f'_c{CONSIST_WEIGHT:g}'

print('cfg          :', MODEL_CFG)
print('seed         :', SEED)
print('epochs       :', EPOCHS)
print('USE_HYPERMIL :', USE_HYPERMIL, f'(weight={MIL_WEIGHT}, hidden={MIL_HIDDEN})' if USE_HYPERMIL else '')
print('run_name     :', RUN_NAME)
print('project      :', RUN_PROJECT)

## 7. Seed + SDP kernel + W&B init

In [ ]:
import random, numpy as np

# Stable SDP kernel (avoid Flash/MEM-efficient mismatch)
os.environ['PYTORCH_SDP_KERNEL'] = 'math'
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)

# Seed (per fold akan di-reseed di train loop)
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# Disable Ultralytics built-in W&B callback (we log per-fold manually)
from ultralytics.utils import SETTINGS
SETTINGS.update({'wandb': False})

print(f'Seed: {SEED}  SDP kernel: math (stable)')
print(f'W&B init akan dibuat per-fold di cell training berikutnya.')

## 8. Auto-download pretrained yolov13s.pt

In [ ]:
from urllib.request import urlretrieve

pt = Path(PRETRAINED)
if not pt.exists():
    url = f'https://github.com/iMoonLab/yolov13/releases/download/yolov13/{PRETRAINED}'
    print(f'Downloading {url}')
    urlretrieve(url, pt)
print(f'Pretrained: {pt}  ({pt.stat().st_size/1e6:.1f} MB)')

## 9. Train - `model.train()` eksplisit (mirror yolo12.ipynb hyperparams)

In [ ]:
import time
from ultralytics import YOLO

# === Training loop over folds (or single split) ===
# Each fold trains a fresh model from PRETRAINED, evals on its own val split.
# All folds logged to W&B as separate runs (grouped by RUN_NAME for 5fold).

fold_results = []   # list of dict per fold with metrics
all_save_dirs = []
NWD_RATIO = 0.0
NWD_C = 12.5
for fold_idx, fold_yaml in enumerate(DATA_YAMLS):
    n_folds = len(DATA_YAMLS)
    is_kfold = SPLIT_MODE == '5fold'

    fold_run_name = f'{RUN_NAME}_fold{fold_idx}' if is_kfold else RUN_NAME
    fold_train_dir = f'{fold_run_name}_train'

    # Re-seed per fold (same SEED for reproducibility; randomness comes from fold data)
    random.seed(SEED); np.random.seed(SEED)
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

    print('\n' + '=' * 70)
    print(f'  FOLD {fold_idx+1}/{n_folds}: {fold_run_name}')
    print(f'  data.yaml: {fold_yaml}')
    print('=' * 70)

    # W&B per fold
    fold_run = wandb.init(
        project=WANDB_PROJECT,
        name=fold_run_name,
        group=RUN_NAME if is_kfold else None,
        reinit=True,
        config=dict(
            model_cfg=MODEL_CFG, data_yaml=fold_yaml, pretrained=PRETRAINED,
            seed=SEED, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
            optimizer='SGD', lr0=0.01, momentum=0.937, cos_lr=True,
            split_mode=SPLIT_MODE, split_seed=SPLIT_SEED,
            fold_idx=fold_idx, n_folds=n_folds,
            nwd_ratio=NWD_RATIO, nwd_c=NWD_C,
            use_hypermil=USE_HYPERMIL, mil_weight=MIL_WEIGHT if USE_HYPERMIL else 0.0,
        ),
        tags=[Path(MODEL_CFG).stem, f'seed{SEED}', SPLIT_MODE,
              f'fold{fold_idx}'] + ([f'nwd{NWD_RATIO}'] if NWD_RATIO > 0 else []),
    )
    print(f'  W&B: {fold_run.url}')

    # Fresh model + pretrained per fold
    model = YOLO(MODEL_CFG)
    try:
        model.load(PRETRAINED)
        print(f'  Loaded pretrained: {PRETRAINED}')
    except Exception as e:
        print(f'  [warn] could not load pretrained: {e}')

    # HyperMIL callback (still WIP; not recommended for now)
    if USE_HYPERMIL:
        sys.path.insert(0, str(REPO_DIR))
        from afb_yolov13 import make_hypermil_callback
        model.add_callback(
            'on_pretrain_routine_start',
            make_hypermil_callback(mil_weight=MIL_WEIGHT, mil_hidden=MIL_HIDDEN,
                                   consist_weight=CONSIST_WEIGHT),
        )

    # Train
    t0 = time.time()
    results = model.train(
        data=str(fold_yaml),
        freeze=2,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        optimizer='SGD',
        lr0=0.01, lrf=0.01,
        momentum=0.937, weight_decay=0.0005,
        cos_lr=True,
        nwd_ratio=NWD_RATIO,
        nwd_c=NWD_C,
        close_mosaic=10,
        hsv_h=0.1, hsv_s=0.3, hsv_v=0.3,
        degrees=30, translate=0.05, scale=0.1,
        flipud=0.3,
        mosaic=0.2, mixup=0.2,
        patience=0,
        amp=True,
        deterministic=True,
        seed=SEED,
        workers=8,
        project=RUN_PROJECT,
        name=fold_train_dir,
        exist_ok=True, save=True, verbose=True,
    )
    train_secs = time.time() - t0
    print(f'\n  Fold {fold_idx} train time: {train_secs/60:.1f} min')
    print(f'  Save dir            : {results.save_dir}')
    all_save_dirs.append(results.save_dir)

    # Eval di val split fold ini (gak ada test split di kfold mode)
    best_pt = Path(results.save_dir) / 'weights' / 'best.pt'
    eval_split = 'test' if SPLIT_MODE == 'single' else 'val'
    eval_model = YOLO(str(best_pt))
    eva = eval_model.val(data=str(fold_yaml), split=eval_split,
                         imgsz=IMGSZ, device=DEVICE, verbose=False)

    map50   = float(eva.box.map50)
    map5095 = float(eva.box.map)
    precision = float(np.mean(np.atleast_1d(eva.box.p)))
    recall    = float(np.mean(np.atleast_1d(eva.box.r)))

    map_at_09 = float('nan')
    try:
        ap_all = eva.box.all_ap
        if ap_all is not None and len(ap_all):
            ap = (ap_all.mean(axis=0) if (hasattr(ap_all, 'ndim') and ap_all.ndim == 2)
                  else ap_all)
            if len(ap) >= 9: map_at_09 = float(ap[8])
    except Exception:
        pass

    fold_m = dict(
        fold=fold_idx,
        eval_split=eval_split,
        mAP50=map50, mAP50_95=map5095, mAP_at_09=map_at_09,
        precision=precision, recall=recall,
        train_min=train_secs / 60,
        save_dir=str(results.save_dir),
    )
    fold_results.append(fold_m)

    print(f'\n  === FOLD {fold_idx} RESULTS ({eval_split}) ===')
    print(f'    mAP50    : {map50:.4f}')
    print(f'    mAP50-95 : {map5095:.4f}')
    print(f'    mAP@0.9  : {map_at_09:.4f}')
    print(f'    P / R    : {precision:.4f} / {recall:.4f}')

    # Log fold summary to W&B
    fold_run.summary[f'{eval_split}/mAP50']      = map50
    fold_run.summary[f'{eval_split}/mAP50-95']   = map5095
    fold_run.summary[f'{eval_split}/mAP@0.9']    = map_at_09
    fold_run.summary[f'{eval_split}/precision']  = precision
    fold_run.summary[f'{eval_split}/recall']     = recall
    fold_run.summary['train/time_min']           = train_secs / 60
    fold_run.summary['fold_idx']                 = fold_idx

    # Upload per-fold plots
    for img in Path(results.save_dir).glob('*.png'):
        tag = img.stem.lower()
        if any(t in tag for t in ('results', 'confusion', 'f1_curve',
                                  'pr_curve', 'p_curve', 'r_curve')):
            try:
                fold_run.log({f'plots/{img.stem}': wandb.Image(str(img))})
            except Exception:
                pass

    fold_run.finish()
    print(f'  W&B fold {fold_idx} finalised.')

print('\n' + '=' * 70)
print(f'  ALL {len(DATA_YAMLS)} FOLD(S) DONE')
print('=' * 70)

## (Removed) Per-epoch curve cell

Per-epoch metrics dari `results.csv` sekarang di-handle per-fold di cell training di atas. Cell ini di-skip — biarkan default Ultralytics save CSV ke `runs/`.

In [ ]:
# Per-epoch curve logging dipindahkan ke training loop (per-fold via results.csv).
# Cell ini sengaja kosong supaya numbering selanjutnya tidak berubah.
print('(per-epoch curves: lihat results.csv di setiap save_dir per fold)')
for fr in fold_results:
    print(f'  fold {fr["fold"]}: {Path(fr["save_dir"]) / "results.csv"}')

## 11. Aggregate metrics across folds (mean ± std) + log summary ke W&B

In [ ]:
import json

print('\n' + '=' * 70)
print(f'  AGGREGATE — {len(fold_results)} fold(s)')
print('=' * 70)

metrics_keys = ['mAP50', 'mAP50_95', 'mAP_at_09', 'precision', 'recall', 'train_min']
agg = {}
for k in metrics_keys:
    vals = np.array([m[k] for m in fold_results if not np.isnan(m[k])])
    if len(vals) > 0:
        agg[k] = dict(mean=float(np.mean(vals)), std=float(np.std(vals)),
                      values=[float(v) for v in vals])

# Tabel ringkas
print(f"\n  {'Metric':<14} {'Mean':>10} {'Std':>10}  {'Per-fold values'}")
print('  ' + '-' * 60)
for k in metrics_keys:
    if k not in agg:
        continue
    vals_str = '  '.join(f'{v:.4f}' for v in agg[k]['values'])
    print(f"  {k:<14} {agg[k]['mean']:>10.4f} {agg[k]['std']:>10.4f}  [{vals_str}]")

# Log summary run ke W&B
print('\n  Logging aggregate run to W&B...')
agg_run = wandb.init(
    project=WANDB_PROJECT,
    name=f'{RUN_NAME}_AGG',
    group=RUN_NAME if SPLIT_MODE == '5fold' else None,
    reinit=True,
    config=dict(
        model_cfg=MODEL_CFG, pretrained=PRETRAINED, seed=SEED,
        epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        split_mode=SPLIT_MODE, n_folds=len(fold_results),
        nwd_ratio=NWD_RATIO, nwd_c=NWD_C,
    ),
    tags=[Path(MODEL_CFG).stem, f'seed{SEED}', SPLIT_MODE, 'aggregate'],
)
for k, v in agg.items():
    agg_run.summary[f'{k}/mean'] = v['mean']
    agg_run.summary[f'{k}/std']  = v['std']
    for i, val in enumerate(v['values']):
        agg_run.summary[f'{k}/fold{i}'] = val
agg_run.summary['n_folds'] = len(fold_results)
agg_run.finish()
print(f'  W&B aggregate run finalised: {RUN_NAME}_AGG')

# Save local JSON for paper / reference
agg_path = Path(all_save_dirs[0]).parent / f'{RUN_NAME}_aggregate.json'
agg_path.write_text(json.dumps(
    dict(run_name=RUN_NAME, model_cfg=MODEL_CFG, split_mode=SPLIT_MODE,
         n_folds=len(fold_results), seed=SEED, epochs=EPOCHS,
         nwd_ratio=NWD_RATIO, nwd_c=NWD_C,
         per_fold=fold_results, aggregate=agg),
    indent=2,
), encoding='utf-8')
print(f'  JSON saved: {agg_path}')

# Set best_pt to fold 0 for downstream diagnostic cells
best_pt = Path(fold_results[0]['save_dir']) / 'weights' / 'best.pt'
print(f'\n  Downstream diagnostic cells will use fold 0 best.pt: {best_pt}')

## 12. Quick predict sample

In [ ]:
# Quick predict sample using fold 0 best.pt
eval_model = YOLO(str(best_pt))

# Source: fold 0 val (or test for single mode)
if SPLIT_MODE == 'single':
    pred_source = f'{SPLIT_DIR}/test/images'
else:
    pred_source = f'{SPLIT_DIR}/fold_0/val/images'

preds = eval_model.predict(
    source=pred_source,
    save=True, imgsz=IMGSZ, conf=0.25, device=DEVICE,
)
print('Predictions saved to:', preds[0].save_dir if preds else None)

## 13. Diagnose baseline (CLI script call)

Output: per-IoU mAP, FP composition, FullPAD gate, HyperACE magnitude, recommendation. JSON disimpan untuk reference berikutnya.

In [ ]:
# Diagnose pakai fold 0 (kalau 5fold) atau single split
DATA_YAML_FOR_DIAG = DATA_YAMLS[0]
DIAG_OUT = (Path('/content') if IS_COLAB else REPO_DIR) / f'diag_{RUN_NAME}_fold0_val'
cmd = (
    f'python "{REPO_DIR}/scripts/diagnose_baseline.py" '
    f'--ckpt "{best_pt}" '
    f'--data "{DATA_YAML_FOR_DIAG}" '
    f'--split val --imgsz {IMGSZ} --device {DEVICE} '
    f'--out "{DIAG_OUT}"'
)
print(cmd, '\n')
os.system(cmd)

import json
js = DIAG_OUT / 'diagnose.json'
if js.exists():
    s = json.loads(js.read_text())
    print('\n=== Recommendations ===')
    for r in s['recommendations']:
        print(f'  [{r["severity"]:>6}] {r["tag"]}: {r["reason"]}')

## 14. Inline probe - FullPAD_Tunnel gates + HyperACE magnitude

Verifikasi langsung apakah pathway HyperACE+FullPAD aktif setelah training:
- `gate ~= 0` -> alpha-trap, pathway tidak kontribusi -> sinyal arsitektur improvement (replace scalar gate, atau init lebih tinggi).
- `gate aktif (|g| > 0.05)` -> HyperACE memang dipakai, novelty arsitektur bisa fokus ke mekanisme di dalamnya.

In [ ]:
from ultralytics.nn.modules.block import FullPAD_Tunnel, HyperACE

probe_model = YOLO(str(best_pt))
m = probe_model.model.cuda().eval()

print('\n=== FullPAD_Tunnel gate values ===')
gates = []
for mod in m.modules():
    if isinstance(mod, FullPAD_Tunnel):
        g = mod.gate.detach().cpu().item()
        gates.append(g)
        status = 'ACTIVE' if abs(g) > 0.05 else ('marginal' if abs(g) > 0.01 else 'DEAD (alpha-trap)')
        print(f'  FullPAD #{len(gates):2d}  gate = {g:+.6f}   [{status}]')
if gates:
    print(f'\n  Mean |gate|: {sum(abs(g) for g in gates)/len(gates):.6f}')
    print(f'  Dead gates : {sum(1 for g in gates if abs(g)<0.01)}/{len(gates)}')

# HyperACE output magnitude
print('\n=== HyperACE output magnitude ===')
hyperace_outs = {}
handles = []
def make_hook(name):
    def fn(module, inp, out):
        hyperace_outs[name] = out.detach().abs().mean().item()
    return fn
for i, mod in enumerate(m.model):
    if isinstance(mod, HyperACE):
        handles.append(mod.register_forward_hook(make_hook(f'layer{i}')))
dummy = torch.randn(1, 3, IMGSZ, IMGSZ).cuda()
with torch.no_grad():
    _ = m(dummy)
for h in handles: h.remove()
for k, v in hyperace_outs.items():
    print(f'  {k}: |output|_mean = {v:.4e}')

## 15. Label quality probe (high-conf FP visual judgment)

Hipotesis: pada dataset AFB phone-camera, mungkin ada **bacilli yang GT miss-label** (Makerere annotation tidak 100% complete). Kalau benar, **model bisa correct tapi disebut FP** -> mAP50 ceiling artifisial.

Output: 30 crop high-conf FP yang jauh dari semua GT. Lo manual judge -> hitung % REAL_BACILLI.
- `> 50% REAL` -> label noise = ceiling -> paper pivot ke 'label quality study' atau pakai dataset lain.
- `20-50% REAL` -> campuran, masih bisa argue mAP50 underestimate.
- `< 20% REAL` -> model genuinely confuses smear/debris -> arch lift mustahil di dataset ini, reframe ke recall.

In [ ]:
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

# Use fold 0 val (or single split val) for label quality probe
if SPLIT_MODE == 'single':
    VAL_IMG_DIR = Path(SPLIT_DIR) / 'val' / 'images'
    VAL_LBL_DIR = Path(SPLIT_DIR) / 'val' / 'labels'
else:
    VAL_IMG_DIR = Path(SPLIT_DIR) / 'fold_0' / 'val' / 'images'
    VAL_LBL_DIR = Path(SPLIT_DIR) / 'fold_0' / 'val' / 'labels'

OUT_DIR     = (Path('/content') if IS_COLAB else REPO_DIR) / 'label_quality_probe'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONF_HIGH = 0.5
DIST_FAR  = 2.0
N_INSPECT = 30

high_conf_fps = []
for img_path in sorted(VAL_IMG_DIR.glob('*.jpg')):
    pil = Image.open(img_path).convert('RGB')
    W, H = pil.size
    res = eval_model.predict(str(img_path), conf=CONF_HIGH, iou=0.6, verbose=False, device=DEVICE)[0]
    if not len(res.boxes):
        continue
    pred = res.boxes.xyxy.cpu().numpy()
    pred_conf = res.boxes.conf.cpu().numpy()

    lp = VAL_LBL_DIR / (img_path.stem + '.txt')
    gt_centers, gt_diams = [], []
    if lp.exists():
        for ln in lp.read_text().strip().splitlines():
            parts = ln.split()
            if len(parts) >= 5:
                _, cx, cy, w, h = map(float, parts[:5])
                gt_centers.append([cx*W, cy*H])
                gt_diams.append(np.sqrt((w*W)*(h*H)))
    gt_centers = np.array(gt_centers) if gt_centers else np.empty((0,2))
    gt_diams   = np.array(gt_diams)   if gt_diams   else np.empty(0)

    for i, (x1,y1,x2,y2) in enumerate(pred):
        pc = np.array([(x1+x2)/2, (y1+y2)/2])
        if len(gt_centers) == 0:
            d_norm = float('inf')
        else:
            d = np.linalg.norm(gt_centers - pc, axis=1)
            j = d.argmin()
            d_norm = float(d[j] / max(gt_diams[j], 1))
        if d_norm > DIST_FAR:
            high_conf_fps.append(dict(img=img_path.name, box=(int(x1),int(y1),int(x2),int(y2)),
                                     conf=float(pred_conf[i]), dist=d_norm))

high_conf_fps.sort(key=lambda x: -x['conf'])
print(f'Total high-conf hard-neg FPs: {len(high_conf_fps)}')

n = min(N_INSPECT, len(high_conf_fps))
cols, rows = 6, (n + 5) // 6
fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*3))
axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
for i, fp in enumerate(high_conf_fps[:n]):
    pil = Image.open(VAL_IMG_DIR / fp['img']).convert('RGB')
    W, H = pil.size
    x1,y1,x2,y2 = fp['box']
    pad = 40
    cx1, cy1 = max(0, x1-pad), max(0, y1-pad)
    cx2, cy2 = min(W, x2+pad), min(H, y2+pad)
    crop = pil.crop((cx1, cy1, cx2, cy2)).copy()
    draw = ImageDraw.Draw(crop)
    draw.rectangle([x1-cx1, y1-cy1, x2-cx1, y2-cy1], outline='red', width=2)
    axes[i].imshow(crop)
    axes[i].set_title(f'#{i+1} conf={fp["conf"]:.2f}\n{fp["img"][:18]}', fontsize=7)
    axes[i].axis('off')
for ax in axes[n:]:
    ax.axis('off')
plt.tight_layout()
plt.savefig(OUT_DIR / 'top_high_conf_fps.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'\nSaved: {OUT_DIR / "top_high_conf_fps.png"}')
print('\nManual judgment - hitung % REAL_BACILLI / 30 -> kasih tau angkanya.')

In [ ]:
# === DEBUG HyperMIL — 1-minute diagnostic ===
import sys, torch
import types as T
sys.path.insert(0, str(REPO_DIR))
from afb_yolov13.hypermil import install_hypermil, _HYPERACE_OUTPUTS, HyperMILLoss
from ultralytics import YOLO

dbg = YOLO('yolov13s.yaml')
dbg.load('yolov13s.pt')
dbg_model = dbg.model.cuda().train()

# Install MIL
print('--- 1. Install ---')
install_hypermil(dbg_model, mil_weight=0.5, mil_hidden=128)

# Manually set args (criterion needs this; trainer normally sets it)
dbg_model.args = T.SimpleNamespace(box=7.5, cls=0.5, dfl=1.5)

# Check state
print('\n--- 2. Post-install state ---')
print(f'  class               : {type(dbg_model).__name__}')
print(f'  _hypermil_hyperace_id: {getattr(dbg_model, "_hypermil_hyperace_id", "MISSING")}')
print(f'  has mil_head        : {hasattr(dbg_model, "mil_head")}')
print(f'  mil_weight          : {getattr(dbg_model, "mil_weight", "MISSING")}')
print(f'  init_criterion bound: {dbg_model.init_criterion.__qualname__}')

# Fake forward
print('\n--- 3. Forward with fake batch ---')
B = 4
imgs = torch.randn(B, 3, 640, 640).cuda()
print(f'  _HYPERACE_OUTPUTS before forward: keys={list(_HYPERACE_OUTPUTS.keys())}')
preds = dbg_model(imgs)
print(f'  _HYPERACE_OUTPUTS after forward : keys={list(_HYPERACE_OUTPUTS.keys())}')
hid = dbg_model._hypermil_hyperace_id
if hid in _HYPERACE_OUTPUTS:
    feat = _HYPERACE_OUTPUTS[hid]
    print(f'  feat shape           : {tuple(feat.shape)}  dtype={feat.dtype}  device={feat.device}')
else:
    print(f'  [BUG] hook did NOT write to expected id {hid}')

# Build criterion
print('\n--- 4. Criterion ---')
crit = dbg_model.init_criterion()
print(f'  criterion class      : {type(crit).__name__}')
print(f'  has base (v8 loss)   : {hasattr(crit, "base")}')
print(f'  has model_ref        : {hasattr(crit, "model_ref")}')
print(f'  model_ref is dbg_model: {crit.model_ref is dbg_model}')

# Fake batch (proper format for v8DetectionLoss)
batch = {
    'img': imgs,
    'batch_idx': torch.tensor([0,0,1,2,2,2,3], dtype=torch.float32).cuda(),
    'cls': torch.zeros(7, 1).cuda(),
    'bboxes': torch.rand(7, 4).cuda(),
}
print('\n--- 5. Loss call (training mode, grad enabled) ---')
print(f'  torch.is_grad_enabled(): {torch.is_grad_enabled()}')
print(f'  dbg_model.training     : {dbg_model.training}')

try:
    loss, items = crit(preds, batch)
    print(f'  total loss           : {loss.item():.4f}')
    print(f'  items                : {items}')
    print(f'  _last_mil_loss       : {dbg_model._last_mil_loss}')
    print(f'  _last_mil_count_mean : {dbg_model._last_mil_count_mean}')
    print(f'  _last_mil_target_mean: {dbg_model._last_mil_target_mean}')
    if dbg_model._last_mil_loss == 0.0:
        print('  [DIAGNOSIS] MIL guard skipped — check above which guard fired')
    else:
        print('  [OK] MIL computed successfully')
except Exception as e:
    print(f'  [ERROR] {type(e).__name__}: {e}')
    import traceback; traceback.print_exc()


## 16. Self-training pseudo-label augmentation (auto-run, no manual steps)

Following Noisy Student (Xie et al. 2020) and STAC (Sohn et al. 2020).

**Workflow (auto, just run):**
1. Generate pseudo-labels per fold from baseline best.pt (~5 min)
2. Visualize 9 samples for sanity check
3. Train 5-fold student on augmented data (~85 min)
4. Aggregate + paired comparison vs baseline

**Defaults:** conf>=0.7, dist>=1.0 box-diameter, single iteration.

**Prerequisite:** baseline 5-fold sudah trained (folders di `RUN_PROJECT/yolov13s_seed42_60ep_foldX_train/`).

In [ ]:
# === ALL-IN-ONE: generate pseudo-labels + viz sanity + train student 5-fold + compare ===

# --- Config (default conservative, paper-defensible) ---
SELF_CONF             = 0.7        # confidence threshold for pseudo-labels
SELF_DIST             = 1.0        # min distance from existing GT (box-diameters)
TEACHER_RUN_NAME      = 'yolov13s_seed42_60ep'   # baseline run_name (cell #14 default)
STUDENT_MODEL_CFG     = 'yolov13s.yaml'          # student arch (same as baseline)
STUDENT_EPOCHS        = 60
STUDENT_NWD_RATIO     = 0.5        # combine with NWD 0.5 (proven +0.6% mAP50)
STUDENT_NWD_C         = 12.8
SKIP_TRAINING         = False      # set True to only generate viz, skip training

# === Step 1: generate pseudo-labels per fold ===
SELF_SPLIT_DIR = (f'/content/tb_5fold_self_iter1_conf{SELF_CONF:g}' if IS_COLAB
                  else f'D:/datasets/tb_5fold_self_iter1_conf{SELF_CONF:g}')

print('=' * 70)
print(f'  STEP 1: generate pseudo-labels (conf>={SELF_CONF}, dist>={SELF_DIST}d)')
print('=' * 70)

self_data_yamls = []
for fold_idx in range(len(DATA_YAMLS)):
    src_split = Path(DATA_YAMLS[fold_idx]).parent
    teacher_pt = Path(RUN_PROJECT) / f'{TEACHER_RUN_NAME}_fold{fold_idx}_train/weights/best.pt'
    out_dir = Path(SELF_SPLIT_DIR) / f'fold_{fold_idx}'

    if not teacher_pt.exists():
        print(f'\n[FATAL] Fold {fold_idx} teacher not found: {teacher_pt}')
        print(f'Did baseline 5-fold finish? Adjust TEACHER_RUN_NAME if your baseline used a different run_name.')
        raise FileNotFoundError(teacher_pt)

    print(f'\n--- Fold {fold_idx} ---')
    if (out_dir / 'data.yaml').exists():
        print(f'  [skip] augmented dataset already exists: {out_dir}')
    else:
        cmd = (
            f'python "{REPO_DIR}/scripts/self_train.py" '
            f'--src-split "{src_split}" '
            f'--teacher-ckpt "{teacher_pt}" '
            f'--out "{out_dir}" '
            f'--conf {SELF_CONF} --dist {SELF_DIST} --device {DEVICE}'
        )
        os.system(cmd)
    self_data_yamls.append(str(out_dir / 'data.yaml'))

print(f'\n{len(self_data_yamls)} augmented data.yaml generated.')

# === Step 2: visualize 9 samples with pseudo-labels (fold 0) ===
print('\n' + '=' * 70)
print(f'  STEP 2: sanity check viz (fold 0, 9 samples)')
print('=' * 70)

from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

aug_train_dir = Path(SELF_SPLIT_DIR) / 'fold_0' / 'train'
orig_train_dir = Path(SPLIT_DIR) / 'fold_0' / 'train'

samples = []
for orig_lbl in (orig_train_dir / 'labels').glob('*.txt'):
    aug_lbl = aug_train_dir / 'labels' / orig_lbl.name
    if not aug_lbl.exists():
        continue
    n_orig = len([ln for ln in orig_lbl.read_text().strip().splitlines() if ln.strip()])
    n_aug = len([ln for ln in aug_lbl.read_text().strip().splitlines() if ln.strip()])
    if n_aug > n_orig:
        samples.append((orig_lbl.stem, n_orig, n_aug))

print(f'  {len(samples)} train imgs got pseudo-labels added')

if samples:
    samples.sort(key=lambda x: -(x[2] - x[1]))
    pick = samples[:9]
    fig, axes = plt.subplots(3, 3, figsize=(15, 15))
    for ax, (stem, n_orig, n_aug) in zip(axes.flatten(), pick):
        img_path = orig_train_dir / 'images' / f'{stem}.jpg'
        if not img_path.exists():
            ax.axis('off'); continue
        pil = Image.open(img_path).convert('RGB').copy()
        draw = ImageDraw.Draw(pil)
        W, H = pil.size
        # Original GT = GREEN
        for ln in (orig_train_dir / 'labels' / f'{stem}.txt').read_text().strip().splitlines():
            p = ln.split()
            if len(p) >= 5:
                _, cx, cy, w, h = map(float, p[:5])
                draw.rectangle([(cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H],
                               outline='lime', width=3)
        # Pseudo = RED (appended after originals)
        aug_lines = [ln for ln in (aug_train_dir / 'labels' / f'{stem}.txt').read_text().strip().splitlines() if ln.strip()]
        for ln in aug_lines[n_orig:]:
            p = ln.split()
            if len(p) >= 5:
                _, cx, cy, w, h = map(float, p[:5])
                draw.rectangle([(cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H],
                               outline='red', width=3)
        ax.imshow(pil)
        ax.set_title(f'{stem[:25]}\nGT={n_orig} +pseudo={n_aug-n_orig}', fontsize=9)
        ax.axis('off')
    plt.suptitle(f'GREEN=GT, RED=Pseudo (conf>={SELF_CONF}, dist>={SELF_DIST}d)',
                 fontsize=14)
    plt.tight_layout()
    viz_path = Path('/content' if IS_COLAB else REPO_DIR) / f'self_train_viz_conf{SELF_CONF:g}.png'
    plt.savefig(viz_path, dpi=100, bbox_inches='tight')
    plt.show()
    print(f'  Viz saved: {viz_path}')
    print('  Cek: red box harus di rod-shape pink/magenta. Kalau di area polos → naikkan SELF_CONF.')
else:
    print('  [WARN] No pseudo-labels generated. Threshold may be too high.')

if SKIP_TRAINING:
    print('\nSKIP_TRAINING=True → stop here. Set False to train student 5-fold.')
else:
    print('\n  Lanjut auto-train student 5-fold di cell berikutnya...')

In [ ]:
# === Step 3: train student 5-fold on augmented data + aggregate + compare ===
# Reuses same hyperparams as baseline 5-fold (for fair comparison).
# Output: student fold_results + aggregate + paired comparison vs baseline.

if SKIP_TRAINING:
    print('SKIP_TRAINING=True → student training dilewati.')
else:
    import time
    from ultralytics import YOLO

    STUDENT_RUN_NAME = f'{Path(STUDENT_MODEL_CFG).stem}_seed{SEED}_{STUDENT_EPOCHS}ep_self_conf{SELF_CONF:g}_nwd{STUDENT_NWD_RATIO:g}'
    print('=' * 70)
    print(f'  STEP 3: train student 5-fold ({STUDENT_RUN_NAME})')
    print(f'  Using NWD ratio {STUDENT_NWD_RATIO} (proven +0.6% mAP50 baseline)')
    print('=' * 70)

    student_results = []
    student_save_dirs = []

    for fold_idx, fold_yaml in enumerate(self_data_yamls):
        random.seed(SEED); np.random.seed(SEED)
        torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
        fold_run_name = f'{STUDENT_RUN_NAME}_fold{fold_idx}'

        print('\n' + '=' * 70)
        print(f'  STUDENT FOLD {fold_idx+1}/5: {fold_run_name}')
        print('=' * 70)

        s_run = wandb.init(
            project=WANDB_PROJECT, name=fold_run_name,
            group=STUDENT_RUN_NAME, reinit=True,
            config=dict(
                model_cfg=STUDENT_MODEL_CFG, data_yaml=fold_yaml,
                pretrained=PRETRAINED, seed=SEED, epochs=STUDENT_EPOCHS,
                imgsz=IMGSZ, batch=BATCH,
                split_mode='5fold_self_iter1', fold_idx=fold_idx,
                self_conf=SELF_CONF, self_dist=SELF_DIST,
                nwd_ratio=STUDENT_NWD_RATIO, nwd_c=STUDENT_NWD_C,
            ),
            tags=[Path(STUDENT_MODEL_CFG).stem, f'seed{SEED}', '5fold_self',
                  f'fold{fold_idx}', f'conf{SELF_CONF:g}', f'nwd{STUDENT_NWD_RATIO:g}'],
        )
        print(f'  W&B: {s_run.url}')

        model = YOLO(STUDENT_MODEL_CFG)
        try:
            model.load(PRETRAINED)
        except Exception as e:
            print(f'  [warn] could not load pretrained: {e}')

        t0 = time.time()
        results = model.train(
            data=str(fold_yaml),
            freeze=2, epochs=STUDENT_EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
            optimizer='SGD', lr0=0.01, lrf=0.01,
            momentum=0.937, weight_decay=0.0005, cos_lr=True,
            nwd_ratio=STUDENT_NWD_RATIO, nwd_c=STUDENT_NWD_C,
            close_mosaic=10,
            hsv_h=0.1, hsv_s=0.3, hsv_v=0.3,
            degrees=30, translate=0.05, scale=0.1,
            flipud=0.3, mosaic=0.2, mixup=0.2,
            patience=0, amp=True, deterministic=True, seed=SEED, workers=8,
            project=RUN_PROJECT, name=f'{fold_run_name}_train',
            exist_ok=True, save=True, verbose=True,
        )
        train_secs = time.time() - t0
        student_save_dirs.append(results.save_dir)

        best_pt_s = Path(results.save_dir) / 'weights' / 'best.pt'
        eva = YOLO(str(best_pt_s)).val(data=str(fold_yaml), split='val',
                                        imgsz=IMGSZ, device=DEVICE, verbose=False)
        m50 = float(eva.box.map50); m5095 = float(eva.box.map)
        p = float(np.mean(np.atleast_1d(eva.box.p)))
        r = float(np.mean(np.atleast_1d(eva.box.r)))
        student_results.append(dict(
            fold=fold_idx, mAP50=m50, mAP50_95=m5095, precision=p, recall=r,
            train_min=train_secs/60, save_dir=str(results.save_dir),
        ))
        print(f'\n  FOLD {fold_idx} STUDENT: mAP50={m50:.4f} mAP50-95={m5095:.4f} '
              f'P={p:.4f} R={r:.4f}')
        s_run.summary['val/mAP50'] = m50
        s_run.summary['val/mAP50-95'] = m5095
        s_run.summary['val/precision'] = p
        s_run.summary['val/recall'] = r
        s_run.finish()

    # === Step 4: aggregate + paired comparison vs baseline ===
    print('\n' + '=' * 70)
    print(f'  STEP 4: STUDENT AGGREGATE + COMPARE vs BASELINE')
    print('=' * 70)

    keys = ['mAP50', 'mAP50_95', 'precision', 'recall']
    print(f"\n  {'Metric':<14} {'Mean':>10} {'Std':>10}  Per-fold")
    print('  ' + '-' * 70)
    student_agg = {}
    for k in keys:
        vals = np.array([m[k] for m in student_results])
        student_agg[k] = dict(mean=float(np.mean(vals)), std=float(np.std(vals)),
                              values=vals.tolist())
        vstr = '  '.join(f'{v:.4f}' for v in vals)
        print(f"  {k:<14} {np.mean(vals):>10.4f} {np.std(vals):>10.4f}  [{vstr}]")

    # Paired comparison vs baseline (if baseline fold_results exists)
    if 'fold_results' in dir() and len(fold_results) == len(student_results):
        print(f'\n  PAIRED COMPARISON vs BASELINE (per-fold delta):')
        for k in ['mAP50', 'mAP50_95']:
            base_vals = np.array([m[k] for m in fold_results])
            stu_vals = np.array([m[k] for m in student_results])
            delta = stu_vals - base_vals
            mean_d = np.mean(delta)
            std_d = np.std(delta, ddof=1) if len(delta) > 1 else 0
            # Paired t-statistic (n-1 df)
            n = len(delta)
            if std_d > 0:
                t = mean_d / (std_d / np.sqrt(n))
            else:
                t = float('nan')
            dstr = '  '.join(f'{d:+.4f}' for d in delta)
            print(f'    Δ {k:<10}: mean={mean_d:+.4f} std={std_d:.4f} t={t:+.2f} '
                  f'(deltas: {dstr})')

    # Log aggregate ke W&B
    agg_run = wandb.init(
        project=WANDB_PROJECT, name=f'{STUDENT_RUN_NAME}_AGG',
        group=STUDENT_RUN_NAME, reinit=True,
        tags=[Path(STUDENT_MODEL_CFG).stem, f'seed{SEED}', '5fold_self', 'aggregate'],
    )
    for k, v in student_agg.items():
        agg_run.summary[f'{k}/mean'] = v['mean']
        agg_run.summary[f'{k}/std']  = v['std']
        for i, val in enumerate(v['values']):
            agg_run.summary[f'{k}/fold{i}'] = val
    agg_run.finish()

    # Save JSON
    import json
    agg_path = Path(student_save_dirs[0]).parent / f'{STUDENT_RUN_NAME}_aggregate.json'
    agg_path.write_text(json.dumps(
        dict(run_name=STUDENT_RUN_NAME, self_conf=SELF_CONF, self_dist=SELF_DIST,
             nwd_ratio=STUDENT_NWD_RATIO, per_fold=student_results,
             aggregate=student_agg), indent=2,
    ), encoding='utf-8')
    print(f'\n  JSON saved: {agg_path}')
    print(f'\n  Compare student mAP50 mean ({student_agg["mAP50"]["mean"]:.4f}) vs '
          f'baseline ({np.mean([m["mAP50"] for m in fold_results]):.4f})')
    print(f'  If Δ > 0.005 with consistent paired delta sign → real signal.')